In [7]:
import pandas as pd
import re
import time
import json
from anthropic import Anthropic


In [ ]:

# ĐỌC DỮ LIỆU ĐÃ TIỀN XỬ LÝ

df = pd.read_csv('C:/Users/user/OneDrive/Desktop/do an co so/DACS/data/data teencode/cleaned.csv')
print(f"Tổng số mẫu: {len(df)}")


Tổng số mẫu: 11916


In [ ]:

# BƯỚC 1: RULE-BASED 

POSITIVE_WORDS = [
    'cảm ơn', 'thank', 'tuyệt', 'hay', 'tốt', 'giỏi', 'xuất sắc',
    'hữu ích', 'chất', 'ok', 'ổn', 'được', 'vui', 'thích', 'yêu',
    '❤', '😊', '🥰', '👍', 'hihi', 'hehe', 'dễ thương', 'pro'
]
NEGATIVE_WORDS = [
    'tệ', 'dở', 'chán', 'khó', 'khổ', 'thất bại', 'không hiểu',
    'rối', 'mệt', 'stress', 'lo', 'sợ', 'buồn', 'thất vọng',
    'tức', 'bực', 'oải', 'nản', '😢', '😭', '😤', 'wtf', 'ugh'
]
QUESTION_PATTERNS = [
    r'\?', r'\bkhông\b', r'\bkh ạ\b', r'\bạ\b.*\?',
    r'\bcó không\b', r'\bcho hỏi\b', r'\bai biết\b', r'\bai có\b',
    r'\bai cho\b', r'\bxin\b', r'\bgiúp\b', r'\bhỏi\b'
]

def rule_based_sentiment(text):
    """Trả về nhãn nếu rõ ràng, None nếu cần AI phán đoán."""
    text_lower = text.lower()

    pos_score = sum(1 for w in POSITIVE_WORDS if w in text_lower)
    neg_score = sum(1 for w in NEGATIVE_WORDS if w in text_lower)
    is_question = any(re.search(p, text_lower) for p in QUESTION_PATTERNS)

    # Ca rõ ràng → gán ngay
    if is_question and pos_score == 0 and neg_score == 0:
        return 'neutral'          # Hỏi đơn thuần
    if pos_score >= 2 and neg_score == 0:
        return 'positive'
    if neg_score >= 2 and pos_score == 0:
        return 'negative'
    if pos_score == 1 and neg_score == 0 and not is_question:
        return 'positive'
    if neg_score == 1 and pos_score == 0 and not is_question:
        return 'negative'

    return None  # Không rõ → chuyển sang AI

# Áp dụng rule-based
df['label'] = df['text'].apply(rule_based_sentiment)

rule_labeled = df['label'].notna().sum()
ai_needed = df['label'].isna().sum()
print(f"Rule-based gán được: {rule_labeled} ({rule_labeled/len(df)*100:.1f}%)")
print(f"Cần AI xử lý:        {ai_needed} ({ai_needed/len(df)*100:.1f}%)")


Rule-based gán được: 5591 (46.9%)
Cần AI xử lý:        6325 (53.1%)


In [ ]:

# BƯỚC 2: CLAUDE API (xử lý batch các ca mơ hồ)
client = Anthropic()

SYSTEM_PROMPT = """Bạn là chuyên gia phân tích cảm xúc văn bản tiếng Việt trong môi trường sinh viên đại học.
Phân loại mỗi văn bản vào đúng 1 trong 3 nhãn:
- positive: vui vẻ, hài lòng, cảm ơn, khích lệ, tích cực
- negative: buồn, lo lắng, bực bội, thất vọng, tiêu cực
- neutral: hỏi thông tin, thông báo, trung lập

Chỉ trả về JSON, không giải thích. Ví dụ:
{"label": "positive"}"""

def classify_batch_ai(texts, indices, batch_size=20, max_retries=3):
    """Gán nhãn batch bằng Claude API."""
    results = {}

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        batch_indices = indices[i:i+batch_size]

        # Tạo prompt cho cả batch
        user_prompt = "Phân loại từng văn bản sau (đánh số từ 0):\n\n"
        for j, text in enumerate(batch_texts):
            user_prompt += f"[{j}] {text[:300]}\n\n"
        user_prompt += f'\nTrả về JSON dạng: {{"results": [{{"id": 0, "label": "..."}}, ...]}}'

        for attempt in range(max_retries):
            try:
                response = client.messages.create(
                    model="claude-sonnet-4-20250514",
                    max_tokens=1000,
                    system=SYSTEM_PROMPT,
                    messages=[{"role": "user", "content": user_prompt}]
                )
                raw = response.content[0].text.strip()
                raw = re.sub(r'```json|```', '', raw).strip()
                data = json.loads(raw)

                for item in data['results']:
                    idx = batch_indices[item['id']]
                    label = item['label']
                    if label not in ('positive', 'negative', 'neutral'):
                        label = 'neutral'
                    results[idx] = label
                break

            except Exception as e:
                print(f"Batch {i//batch_size + 1}, lần thử {attempt+1}: {e}")
                if attempt == max_retries - 1:
                    # Fallback: gán neutral cho cả batch
                    for idx in batch_indices:
                        results[idx] = 'neutral'
                time.sleep(2)

        # Rate limit: nghỉ giữa các batch
        time.sleep(0.5)
        print(f"Đã xử lý {min(i+batch_size, len(texts))}/{len(texts)} mẫu AI...")

    return results

# Lấy các dòng cần AI xử lý
ai_mask = df['label'].isna()
ai_texts = df.loc[ai_mask, 'text'].tolist()
ai_indices = df.loc[ai_mask].index.tolist()

if ai_texts:
    ai_results = classify_batch_ai(ai_texts, ai_indices)
    for idx, label in ai_results.items():
        df.at[idx, 'label'] = label

C:\Users\user\AppData\Local\Temp\ipykernel_28372\1790440069.py:31: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client.messages.create(


Batch 1, lần thử 1: "Could not resolve authentication method. Expected either api_key or auth_token to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"
Batch 1, lần thử 2: "Could not resolve authentication method. Expected either api_key or auth_token to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"
Batch 1, lần thử 3: "Could not resolve authentication method. Expected either api_key or auth_token to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"
Đã xử lý 20/6325 mẫu AI...
Batch 2, lần thử 1: "Could not resolve authentication method. Expected either api_key or auth_token to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"
Batch 2, lần thử 2: "Could not resolve authentication method. Expected either api_key or auth_token to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"
Bat

In [ ]:
# BƯỚC 3: THỐNG KÊ & LƯU

print("\n=== PHÂN PHỐI NHÃN ===")
print(df['label'].value_counts())
print(f"\nTỷ lệ:\n{df['label'].value_counts(normalize=True).mul(100).round(1)}")

# Chỉ giữ 2 cột cần thiết
df = df[['text', 'label']]

print("\n=== PHÂN PHỐI NHÃN ===")
print(df['label'].value_counts())

print(f"\nTỷ lệ:\n{df['label'].value_counts(normalize=True).mul(100).round(1)}")

# Đổi tên cột
df = df.rename(columns={
    'label': 'sentiment',
    'text': 'sentence'
})

# Sắp xếp lại thứ tự cột
df = df[['sentiment', 'sentence']]

# Lưu file
df.to_excel('clean_labeled.xlsx', index=False)

print("Đã lưu file với cột: sentiment | sentence")



=== PHÂN PHỐI NHÃN ===
label
neutral     9892
positive    1744
negative     280
Name: count, dtype: int64

Tỷ lệ:
label
neutral     83.0
positive    14.6
negative     2.3
Name: proportion, dtype: float64

=== PHÂN PHỐI NHÃN ===
label
neutral     9892
positive    1744
negative     280
Name: count, dtype: int64

Tỷ lệ:
label
neutral     83.0
positive    14.6
negative     2.3
Name: proportion, dtype: float64
Đã lưu file với cột: sentiment | sentence
